In [ ]:
!pip install -q langchain langchain-community langchain-groq \
    faiss-cpu pypdf sentence-transformers \
    streamlit pyngrok nest-asyncio \
    langchain-text-splitters

In [ ]:
!pip install langchain==0.1.20

In [ ]:
import os

os.environ["Openai_API_KEY"] = ""
NGROK_AUTH_TOKEN = ""

In [ ]:
app_code = '''
import os
import streamlit as st
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_groq import ChatGroq
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate

st.set_page_config(page_title="🌿 Homeopathy Clinic Assistant", layout="centered")
st.title("🌿 Homeopathy Clinic Assistant")
st.caption("Upload patient PDFs and query appointment schedules")

UPLOAD_DIR = "uploads"
os.makedirs(UPLOAD_DIR, exist_ok=True)

# ── Session State ────────────────────────────────────────────
if "vectorstore" not in st.session_state:
    st.session_state.vectorstore = None
if "uploaded_files" not in st.session_state:
    st.session_state.uploaded_files = []
if "query_input" not in st.session_state:
    st.session_state.query_input = ""

# ── Load Models ──────────────────────────────────────────────
@st.cache_resource
def load_embeddings():
    return HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

@st.cache_resource
def load_llm():
    return ChatGroq(
        model_name="llama-3.3-70b-versatile",   # ✅ updated model
        groq_api_key=os.environ["GROQ_API_KEY"],
        temperature=0.2
    )

embeddings = load_embeddings()
llm = load_llm()

PROMPT_TEMPLATE = """
You are an AI Homeopathy Clinic Assistant. Use only the patient records below.

Rules:
- Chronic cases → follow-up in 2–6 weeks
- Acute cases → follow-up in 2–7 days
- Stable/improving → follow-up in 4–8 weeks
- No visit > 3 months → mark OVERDUE FOLLOW-UP
- Never guess or hallucinate patient data

Always respond in this exact format:

### 🧑 Patient Summary
- Name:
- Age:
- Condition Type:
- Key Symptoms:
- Last Visit Date:

### 📜 Medical History Insight
[brief timeline of visits]

### 📅 Appointment Recommendation
- Suggested Date:
- Reason:
- Urgency Level: Low / Medium / High / Critical

### ⚠️ Alerts
[missed follow-ups, urgent flags, or None]

### 💡 Doctor Notes
[short actionable tip for doctor/compounder]

Context: {context}
Question: {question}
"""

prompt = PromptTemplate(
    input_variables=["context", "question"],
    template=PROMPT_TEMPLATE
)

# ── Upload Section ───────────────────────────────────────────
st.subheader("📁 Upload Patient Records (PDF)")
uploaded_files = st.file_uploader(
    "Upload one or more patient PDF files",
    type="pdf",
    accept_multiple_files=True
)

if uploaded_files:
    for f in uploaded_files:
        if f.name not in st.session_state.uploaded_files:
            path = os.path.join(UPLOAD_DIR, f.name)
            with open(path, "wb") as out:
                out.write(f.read())

            loader = PyPDFLoader(path)
            docs = loader.load()

            splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
            chunks = splitter.split_documents(docs)

            if st.session_state.vectorstore is None:
                st.session_state.vectorstore = FAISS.from_documents(chunks, embeddings)
            else:
                st.session_state.vectorstore.add_documents(chunks)

            st.session_state.uploaded_files.append(f.name)
            st.success(f"✅ {f.name} uploaded — {len(chunks)} chunks indexed")

if st.session_state.uploaded_files:
    st.info(f"📂 Indexed: {', '.join(st.session_state.uploaded_files)}")

st.divider()

# ── Query Section ────────────────────────────────────────────
st.subheader("💬 Ask About a Patient")

# ✅ Fixed buttons — use session_state key directly
col1, col2, col3 = st.columns(3)
with col1:
    if st.button("📋 Overdue patients", use_container_width=True):
        st.session_state.query_input = "List all patients with overdue follow-ups"
with col2:
    if st.button("🚨 Urgent cases", use_container_width=True):
        st.session_state.query_input = "Show all urgent or critical cases"
with col3:
    if st.button("📅 This week", use_container_width=True):
        st.session_state.query_input = "Who needs an appointment this week?"

# ✅ Text input bound to session_state
query = st.text_input(
    "Or type your query:",
    value=st.session_state.query_input,
    placeholder="e.g. Schedule appointment for Rahul Patil",
    key="query_box"
)

if st.button("🔍 Ask", use_container_width=True):
    final_query = query.strip() or st.session_state.query_input.strip()

    if not final_query:
        st.warning("⚠️ Please enter a query.")
    elif st.session_state.vectorstore is None:
        st.warning("⚠️ Please upload at least one patient PDF first.")
    else:
        with st.spinner("Analyzing patient records..."):
            try:
                qa_chain = RetrievalQA.from_chain_type(
                    llm=llm,
                    retriever=st.session_state.vectorstore.as_retriever(
                        search_kwargs={"k": 4}
                    ),
                    chain_type_kwargs={"prompt": prompt}
                )
                result = qa_chain.invoke({"query": final_query})
                st.markdown("---")
                st.markdown(result["result"])
            except Exception as e:
                st.error(f"❌ Error: {str(e)}")
'''

with open("app.py", "w") as f:
    f.write(app_code)

print("✅ app.py written")

✅ app.py written


In [ ]:
import subprocess
import threading
import time
import nest_asyncio
from pyngrok import ngrok, conf

nest_asyncio.apply()
conf.get_default().auth_token = NGROK_AUTH_TOKEN

def run_streamlit():
    subprocess.run([
        "streamlit", "run", "app.py",
        "--server.port", "8501",
        "--server.headless", "true"
    ])

threading.Thread(target=run_streamlit, daemon=True).start()
time.sleep(6)
print("✅ Streamlit running")

public_url = ngrok.connect(8501)
print(f"\n🌐 Open your app: {public_url}")

✅ Streamlit running

🌐 Open your app: NgrokTunnel: "https://gumminess-baggie-thrive.ngrok-free.dev" -> "http://localhost:8501"
